# M1 Monitoring Dashboard — Real-Time Incident Intelligence

## Purpose
Real-time operational dashboard for incident monitoring, SLA tracking, and queue health management.

## Data Sources
- **Primary**: ServiceNow incident export (5000+ records)
- **Fields**: number, created, priority, state, assignment_group, tower, sdm, category, made_sla, mttr_hours
- **Update Frequency**: Real-time (API)

## Key Metrics & Methodology

### 1. KPI Calculation
```
Active Incidents = COUNT(state IN ['Open', 'In Progress', 'On Hold'])
Critical (P1) = COUNT(priority=1 AND state NOT IN ['Resolved', 'Closed'])
High (P2) = COUNT(priority=2 AND state NOT IN ['Resolved', 'Closed'])
SLA Compliance % = (COUNT(made_sla=TRUE) / COUNT(all)) * 100
Avg MTTR = MEAN(mttr_hours) WHERE state IN ['Resolved', 'Closed']
SLA Breaches = COUNT(made_sla=FALSE AND state IN ['Resolved', 'Closed'])
```

### 2. By-Group Distribution
- **Chart Type**: Stacked bar by state (Open, In Progress, On Hold, Resolved, Closed)
- **Methodology**: Group incidents by assignment_group, count by state
- **Use Case**: Queue health per group, workload distribution

### 3. By-Category Analysis
- **Chart Type**: Donut (pie chart variant)
- **Methodology**: Group by category, count total incidents, calculate %
- **Insight**: Service distribution, demand patterns

### 4. SLA KPI Tracking
- **Chart Type**: Radial gauge + table
- **Targets**: P1=4h, P2=8h, P3=72bh, P4=120bh (contractual)
- **Calculation**: (Compliant incidents / Total incidents per priority) * 100
- **Business Impact**: Contract compliance, penalty avoidance

### 5. Priority Heatmap
- **Chart Type**: 2D matrix (assignment_group × priority)
- **Methodology**: Count incidents, normalize per group, color-code intensity
- **Use Case**: Identify which groups handle which priorities, load balancing

## Filters & Context
- **Date Range**: Configurable date picker (default: last 6 months)
- **Tower**: Multi-select (A&I, D&A, DES, SAP)
- **SDM**: Multi-select (8 service delivery managers)
- **Assignment Group**: Multi-select (96+ groups)
- **Priority**: 1=Critical, 2=High, 3=Moderate, 4=Standard
- **Category**: Application Access, Hardware, Network, etc.
- **State**: Open, In Progress, On Hold, Resolved, Closed

## Data Quality Checks
1. **Nullability**: Flag incidents with missing tower/sdm/priority
2. **MTTR Anomalies**: Flag MTTR > 720 hours (30 days) as potential data errors
3. **SLA Consistency**: Validate made_sla bool matches mttr_hours vs target

## Performance Optimization
- **Caching**: 5-minute cache on KPI queries
- **Aggregation**: Pre-calculate by-group/category during data load
- **Pagination**: Incidents table: 50 rows/page


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from datetime import datetime, timedelta

# Load sample data
API_BASE = 'http://127.0.0.1:8002/api'

# Fetch KPIs
response = requests.get(f'{API_BASE}/monitoring/kpis')
kpis = response.json()
print('KPIs:', kpis)

In [ ]:
# Visualization: KPI Cards
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('M1 Monitoring Dashboard KPIs', fontsize=16, fontweight='bold')

metrics = [
    ('Total Active', kpis.get('total_active', 0), 'blue'),
    ('Critical P1', kpis.get('critical_p1', 0), 'red'),
    ('High P2', kpis.get('high_p2', 0), 'orange'),
    ('SLA Compliance %', kpis.get('sla_compliance_pct', 0), 'green'),
    ('Avg MTTR (h)', kpis.get('avg_mttr_hours', 0), 'purple'),
    ('SLA Breaches', kpis.get('sla_breaches', 0), 'darkred'),
]

for idx, (ax, (title, value, color)) in enumerate(zip(axes.flat, metrics)):
    ax.text(0.5, 0.5, str(int(value)), 
            fontsize=40, fontweight='bold', ha='center', va='center', color=color)
    ax.text(0.5, 0.1, title, fontsize=12, ha='center', style='italic')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

plt.tight_layout()
plt.savefig('/tmp/m1_kpis.png', dpi=150, bbox_inches='tight')
plt.show()

print('KPI visualization saved')

## Key Insights from M1 Dashboard

1. **SLA Compliance**: Track against contractual targets (90%+ is target)
2. **P1 Critical**: Should be <5 active at any time (escalate if higher)
3. **By-Group Load**: Identify overloaded groups (queue > 50 for any state)
4. **Category Trend**: Monitor growing categories (may indicate new issues)
5. **MTTR vs Target**: If Avg MTTR > P1 target (4h), escalate immediately
